In [78]:
import pandas as pd
import seaborn as sns

In [79]:
titanic_raw = sns.load_dataset("titanic")
titanic_raw.to_csv("data/raw/titanic.csv", index = False)
print("okayy")

okayy


In [80]:
titanic = pd.read_csv("data/raw/titanic.csv")
print(titanic.shape)
print(titanic.head(3))

(891, 15)
   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  


## 1. 열 선택

In [81]:
# 1-1
fare = titanic["fare"]
print(type(fare))

# 1-2 
result = titanic[["survived", "pclass", "sex", "age"]]
print(type(result))
print(result.shape)

# 1-3
# 열 이름 하나만 주면 Series이고 리스트로 묶어서 주면 DataFrame이 된다 

<class 'pandas.Series'>
<class 'pandas.DataFrame'>
(891, 4)


## 2. loc vs iloc

In [82]:
# 2-1
x = titanic.loc[10:15, ["sex", "age"]]
print(x)


# 2-2
y = titanic.iloc[10:15, [2,3]]
print(y)

# 2-3
print(len(x))
print(len(y))
# loc는 끝 포함, iloc은 끝을 미포함이라서 출력되는 행 수가 다르다

       sex   age
10  female   4.0
11  female  58.0
12    male  20.0
13    male  39.0
14  female  14.0
15  female  55.0
       sex   age
10  female   4.0
11  female  58.0
12    male  20.0
13    male  39.0
14  female  14.0
6
5


## 3.iloc 응용

In [83]:
z = titanic.iloc[-3:, 0:4]
print(z)

     survived  pclass     sex   age
888         0       3  female   NaN
889         1       1    male  26.0
890         0       3    male  32.0


## 4. 조건 마스크 

In [84]:
# 4-1
mask = titanic['age'] < 10 
print(mask.head(5))

# 4-2 

print(len(titanic[mask]))

0    False
1    False
2    False
3    False
4    False
Name: age, dtype: bool
62


## 5. 복합조건

In [85]:
# 5-1
mask = (titanic['pclass']==3) & (titanic['sex']== 'male') & (titanic['survived']== 1)
print(len(titanic[mask])) 

# 5-2
mask = (titanic['pclass']==1) | (titanic['fare']>=100) 
print(len(titanic[mask])) 

# 5-3
mask = (titanic['alone'])
print(len(titanic[~mask])) 

47
216
354


## 6. isin,between

In [86]:
# 6-1
mask = titanic['embark_town'].isin(['Cherbourg', 'Queenstown'])
print(titanic[mask].shape)

# 6-2
mask = titanic['age'].between(20, 30)
print(titanic[mask].shape)
print(titanic[mask]['age'].min())
print(titanic[mask]['age'].max())

# 6-3
mask = titanic['age'].between(20, 30, inclusive="neither")
print(titanic[mask].shape)

(245, 15)
(245, 15)
20.0
30.0
(205, 15)


## 7. loc[조건, 열]

In [87]:
# 7
mask = (titanic['age'] <= 12) & (titanic['pclass'] == 3) 
kids = titanic.loc[mask, ['age', 'sex', 'survived']] 
print(len(kids)) 
print(round(kids['survived'].mean(), 2)) 

48
0.42


## 8. query

In [88]:
# 8-1
min_fare = 50
print(titanic.query("embark_town == 'Cherbourg' & fare >= @min_fare").shape)

# 8-2
print(len(titanic.query("pclass == 3 & sex == 'male' & survived == 1")))

(61, 15)
47


## 9. 정렬

In [89]:
# 9-1
print(titanic.sort_values(by=['pclass', 'fare'], ascending=[True, False])[['pclass', 'fare', 'sex', 'age']].head(3))

# 9-2
x = titanic.sort_values(by='fare', ascending=False)
print(x.iloc[0]['fare'])
print(x.loc[0]['fare'])
# iloc[0]은 정렬 후 첫 번째 행이고, loc[0]은 원래 0번 승객이라서 값이 다르다

     pclass      fare     sex   age
258       1  512.3292  female  35.0
679       1  512.3292    male  36.0
737       1  512.3292    male  35.0
512.3292
7.25


## 10. 종합- 20~40세 남성, 1,2등석 vs 3등석

In [90]:
# 10-1
mask = (titanic['sex'] == 'male') & (titanic['age'].between(20, 40)) & (titanic['pclass'].isin([1, 2]))
y = titanic.loc[mask, ['pclass', 'age', 'fare', 'survived']]

# 10-2
print(len(y))
print(round(y['survived'].mean(), 2))
print(y.sort_values(by='fare', ascending=False).head(3))

# 10-3
print(len(titanic.query("sex == 'male' & age >= 20 & age <= 40 & pclass in [1, 2]")))

# 10-4
mask = (titanic['sex'] == 'male') & (titanic['age'].between(20, 40)) & (titanic['pclass'] == 3)
z = titanic.loc[mask, ['pclass', 'age', 'fare', 'survived']]
print(round(z['survived'].mean(), 2))

103
0.24
     pclass   age      fare  survived
679       1  36.0  512.3292         1
737       1  35.0  512.3292         1
118       1  24.0  247.5208         0
103
0.15


## 10-4 
20~40세 남성 승객들 중에 1,2등석 승객의 생존율이 3등석 승객보다 높다. 비슷한 조건에서 객실 등급에 따라 생존율이 차이가 난다. 높은 등급일수록 생존에 유리하다 
